# Notebook 30 -- Luyben Recycle Plant Model Demo

Deliverables:
1. Verify nominal steady state matches Luyben (1994) Table 1 qualitatively.
2. Demonstrate snowball instability in open-loop with low alpha.
3. Show 5-loop closed-loop response to a step catalyst decay.
4. Verify all 12 scenario configs reach a stable degraded steady state.


In [ ]:
import sys; sys.path.insert(0, '../src')
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from cstr_sbi.luyben.physics import (
    NOMINAL_THETA, NOMINAL_INLET, NOMINAL_CTRL_ALL, NOMINAL_Y0,
    simulate_to_steady_state, simulate_trajectory, extract_observations,
    TSP_R, TSP_S, PARAM_NAMES, STATE_NAMES, OBS_NAMES,
)

print('JAX devices:', jax.devices())
print('Parameter names:', PARAM_NAMES)
print('State names:', STATE_NAMES)


## 1. Nominal steady state

In [ ]:
y_ss = simulate_to_steady_state(NOMINAL_THETA, NOMINAL_INLET, NOMINAL_CTRL_ALL, NOMINAL_Y0)
print('Nominal steady state:')
for name, val in zip(STATE_NAMES, np.asarray(y_ss)):
    print(f'  {name:8s} = {val:.4f}')
print(f'\nT_r setpoint: {TSP_R} K  (error: {abs(float(y_ss[2]) - TSP_R):.3f} K)')
print(f'T_s setpoint: {TSP_S} K  (error: {abs(float(y_ss[7]) - TSP_S):.3f} K)')


## 2. Trajectory under nominal conditions (2-hour window)

In [ ]:
ts, ys = simulate_trajectory(NOMINAL_THETA, NOMINAL_INLET, NOMINAL_CTRL_ALL, y_ss,
                             t_final=120.0, n_save=121)
obs = extract_observations(ys, NOMINAL_THETA, NOMINAL_CTRL_ALL)

fig, axes = plt.subplots(2, 4, figsize=(14, 6), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes.ravel(), OBS_NAMES)):
    ax.plot(np.asarray(ts), np.asarray(obs[:, i]))
    ax.set_xlabel('time [min]'); ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.suptitle('Nominal 2-hour trajectory -- 8 observable channels')
plt.show()


## 3. Snowball demonstration (open-loop, alpha=0.5)

In [ ]:
from cstr_sbi.luyben.physics import luyben_open_loop_rhs
import diffrax

# Severe catalyst decay in open-loop to show snowball buildup
theta_snow = NOMINAL_THETA.at[0].set(0.5)  # alpha=0.5
ts_ol, ys_ol = simulate_trajectory(
    theta_snow, NOMINAL_INLET, NOMINAL_CTRL_ALL, y_ss,
    t_final=240.0, n_save=241
)
obs_ol = extract_observations(ys_ol, theta_snow, NOMINAL_CTRL_ALL)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
axes[0].plot(np.asarray(ts_ol), np.asarray(obs_ol[:, 0]))  # T_r
axes[0].axhline(TSP_R, ls='--', c='C1'); axes[0].set_ylabel('T_r [K]')
axes[1].plot(np.asarray(ts_ol), np.asarray(obs_ol[:, 5]))  # F_R
axes[1].set_ylabel('F_R [L/min] (recycle flow)')
axes[2].plot(np.asarray(ts_ol), np.asarray(obs_ol[:, 2]))  # Qc
axes[2].set_ylabel('Qc [J/min]')
for ax in axes: ax.set_xlabel('time [min]'); ax.grid(alpha=0.3)
fig.suptitle('Snowball demonstration: alpha=0.5, closed-loop response')
plt.show()
print('Max F_R:', np.asarray(obs_ol[:, 5]).max(), '(nominal:', 40.0, ')')


## 4. All 12 scenarios -- steady state check

In [ ]:
from cstr_sbi.luyben.scenarios import list_closed_loop_configs
import pandas as pd

rows = []
for sc in list_closed_loop_configs():
    try:
        y_sc = simulate_to_steady_state(sc.theta(), NOMINAL_INLET)
        status = 'OK' if not np.any(np.isnan(np.asarray(y_sc))) else 'NaN'
    except Exception as e:
        y_sc = np.full(13, np.nan)
        status = f'ERR: {e}'
    obs_sc = extract_observations(y_sc[None, :], sc.theta())[0] if status == 'OK' else np.full(8, np.nan)
    rows.append({
        'id': sc.id, 'name': sc.name,
        'T_r': float(y_sc[2]), 'F_R': float(obs_sc[5]) if status == 'OK' else np.nan,
        'F_prod': float(obs_sc[7]) if status == 'OK' else np.nan,
        'status': status,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))


## 5. nb30 acceptance

| Criterion | Result |
|---|---|
| Nominal SS: T_r -> TSP_R within 1 K | Check above |
| Snowball: F_R increases significantly for low alpha | Check above |
| All 12 scenarios reach SS without NaN | Check above |
